# 00 — Setup e pulizia del progetto

Da eseguire **una volta sola**, prima di tutto il resto. Il notebook:
1. crea l'inventario del progetto attuale (`inventario_progetto.csv`);
2. crea copie leggere (`*_light.ipynb`) dei notebook troppo pesanti;
3. crea la nuova struttura delle cartelle;
4. **sposta** (non cancella) tutto il vecchio materiale in `_archivio/` — prima in modalità prova (`DRY_RUN = True`).

Struttura dopo la pulizia (`F:\Tesi Basket`):
```
notebooks/      00, 09, 10, 11, 12, 13   (i nuovi)
videos/         le 7 clip
annotations/    manifest, annotazioni, tasks (le tue etichette)
outputs/        output SAM2 (vecchi + quelli del notebook 09)
scripts/        codice originale (correttore, valutazione)
models/, configs/, basketball_player_detection_v18/
experiments/    E1_detection, E2_tracking, E3_correzione -> CSV + figure
demo/, tesi/figure/
_archivio/      vecchi notebook, documenti, exports, reports, tests
```

In [1]:
# ==== SETUP COMUNE (uguale in tutti i notebook) ====
import sys, os
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/NBA_Tracking_Project')   # progetto su Drive
    FAST = Path('/content/fast')                                 # disco locale Colab (veloce)
else:
    ROOT = Path(r'F:\Tesi Basket')                               # progetto in locale
    FAST = ROOT / '_cache'
FAST.mkdir(parents=True, exist_ok=True)
assert ROOT.is_dir(), f'Cartella del progetto non trovata: {ROOT}'
print('Progetto:', ROOT)

Progetto: F:\Tesi Basket


## 1. Inventario

In [ ]:
from datetime import datetime
import pandas as pd
rows = []
for p in ROOT.rglob('*'):
    if p.is_file() and '_archivio' not in p.parts:
        st = p.stat()
        rows.append({'path': str(p.relative_to(ROOT)), 'size_mb': round(st.st_size / 1e6, 2),
                     'modified': datetime.fromtimestamp(st.st_mtime).strftime('%Y-%m-%d %H:%M')})
inv = pd.DataFrame(rows).sort_values('path')
inv.to_csv(ROOT / 'inventario_progetto.csv', index=False)
print(len(inv), 'file,', round(inv.size_mb.sum() / 1e3, 2), 'GB')
inv.assign(cartella=inv.path.str.split(r'[\\/]').str[0]).groupby('cartella').size_mb.agg(['count', 'sum']).sort_values('sum', ascending=False)

## 2. Copie leggere dei notebook pesanti (> 5 MB)
Tiene codice e output testuali, toglie immagini e video incorporati. Gli originali non vengono toccati.

In [ ]:
import json
HEAVY = {'image/png', 'image/jpeg', 'image/gif', 'image/svg+xml', 'text/html',
         'application/javascript', 'application/vnd.jupyter.widget-view+json'}
for nb_path in ROOT.rglob('*.ipynb'):
    if nb_path.stem.endswith('_light') or '_archivio' in nb_path.parts or nb_path.stat().st_size < 5e6:
        continue
    d = json.loads(nb_path.read_text(encoding='utf-8'))
    for cell in d['cells']:
        if cell['cell_type'] != 'code': continue
        kept = []
        for out in cell.get('outputs', []):
            if 'data' in out:
                out['data'] = {k: v for k, v in out['data'].items() if k not in HEAVY}
                if not out['data']: continue
            kept.append(out)
        cell['outputs'] = kept
    light = nb_path.with_name(nb_path.stem + '_light.ipynb')
    light.write_text(json.dumps(d, ensure_ascii=False), encoding='utf-8')
    print(f'{nb_path.name}: {nb_path.stat().st_size/1e6:.1f} MB -> {light.stat().st_size/1e6:.2f} MB')

## 3. Riorganizzazione (minima)

Regole pensate per `F:\Tesi Basket`:
- **restano dove sono** (servono ai notebook): `annotations/`, `videos/`, `outputs/`, `models/`, `scripts/`, `configs/`, `basketball_player_detection_v18/`, `.venv-eval/`, `Annota.cmd`, `.gitignore`;
- i nuovi notebook vanno in `notebooks/` (se lo zip è stato scompattato in una sottocartella, vengono portati su);
- **in archivio** `_archivio/<data>/`: vecchi notebook 01–06, documenti `.md` di Codex, `exports/`, `reports/`, `tests/`, `archivio/`.

Prima `DRY_RUN = True`, controlla l'elenco, poi `False`.

In [4]:
import shutil, re
from datetime import datetime
DRY_RUN = False

KEEP = {'annotations', 'videos', 'outputs', 'models', 'scripts', 'configs', 'basketball_player_detection_v18',
        '.venv-eval', '.git', '.gitignore', 'Annota.cmd', 'notebooks', 'experiments', 'demo', 'tesi',
        '_archivio', '_cache', 'datasets', 'README_TESI.md', 'requirements.txt', 'inventario_progetto.csv'}
NEW_NB = re.compile(r'^(0[09]|1[0-9])_.*\.ipynb$')
ARCH = ROOT / '_archivio' / datetime.now().strftime('%Y%m%d_%H%M')

moves = []
# zip scompattato in una sottocartella: porta su il contenuto
for nested in [ROOT / 'NBA_Tracking_Project', ROOT / 'notebooks_tesi']:
    if nested.is_dir():
        for p in nested.rglob('*'):
            if p.is_file() and p.name != '.keep':
                dst = ROOT / p.relative_to(nested)
                if not dst.exists(): moves.append((p, dst))
for p in ROOT.iterdir():
    if p.name in KEEP: continue
    if p.is_file() and NEW_NB.match(p.name):
        moves.append((p, ROOT / 'notebooks' / p.name)); continue
    moves.append((p, ARCH / p.name))
nbdir = ROOT / 'notebooks'
if nbdir.is_dir():
    for p in nbdir.iterdir():
        if not NEW_NB.match(p.name): moves.append((p, ARCH / 'notebooks' / p.name))

for src, dst in moves:
    print(('[prova] ' if DRY_RUN else '') + f'{src.relative_to(ROOT)}  ->  {dst.relative_to(ROOT)}')
if not DRY_RUN:
    for src, dst in moves:
        if src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True); shutil.move(str(src), str(dst))
    for d in ['notebooks', 'experiments/E1_detection', 'experiments/E2_tracking', 'experiments/E3_correzione', 'demo', 'tesi/figure']:
        (ROOT / d).mkdir(parents=True, exist_ok=True)
    print('\nFatto:', len(moves), 'spostamenti. Archivio:', ARCH)

01_NBA_tracking_revisionato.ipynb  ->  _archivio\20261022_1330\01_NBA_tracking_revisionato.ipynb
02_NBA_detection_revisionato.ipynb  ->  _archivio\20261022_1330\02_NBA_detection_revisionato.ipynb
03_NBA_correzione_valutazione.ipynb  ->  _archivio\20261022_1330\03_NBA_correzione_valutazione.ipynb
04_NBA_manifest_annotazioni.ipynb  ->  _archivio\20261022_1330\04_NBA_manifest_annotazioni.ipynb
05_NBA_palla_lavagna.ipynb  ->  _archivio\20261022_1330\05_NBA_palla_lavagna.ipynb
06_NBA_tracking_recupero_giocatori.ipynb  ->  _archivio\20261022_1330\06_NBA_tracking_recupero_giocatori.ipynb
archivio  ->  _archivio\20261022_1330\archivio
AVVIO_COLAB.md  ->  _archivio\20261022_1330\AVVIO_COLAB.md
exports  ->  _archivio\20261022_1330\exports
INIZIA_QUI.md  ->  _archivio\20261022_1330\INIZIA_QUI.md
LINK_UTILI_NOTEBOOK_ROBOFLOW.md  ->  _archivio\20261022_1330\LINK_UTILI_NOTEBOOK_ROBOFLOW.md
notebooks_tesi.zip  ->  _archivio\20261022_1330\notebooks_tesi.zip
PALLA_E_LAVAGNA.md  ->  _archivio\20261022_1

## 4. Controllo finale dei dati necessari

Se manca qualcosa, recuperalo da `_archivio/` (es. CSV di tracking SAM2 dei vecchi output) e rimettilo nel posto indicato.

In [6]:
checks = {'Clip (videos/)': ROOT / 'videos', 'Manifest': ROOT / 'annotations' / 'manifest.csv',
          'Annotazioni': ROOT / 'annotations' / 'annotations.csv', 'Piano sequenze': ROOT / 'annotations' / 'tasks.csv',
          'Script originali': ROOT / 'scripts' / 'height_corrector.py', 'Nuovi notebook': ROOT / 'notebooks'}
for k, p in checks.items():
    print(('OK      ' if p.exists() else 'MANCANTE'), k, '->', p.relative_to(ROOT))
print('Notebook:', sorted(x.name for x in (ROOT / 'notebooks').glob('*.ipynb')) if (ROOT / 'notebooks').is_dir() else '-')

OK       Clip (videos/) -> videos
OK       Manifest -> annotations\manifest.csv
OK       Annotazioni -> annotations\annotations.csv
OK       Piano sequenze -> annotations\tasks.csv
OK       Script originali -> scripts\height_corrector.py
OK       Nuovi notebook -> notebooks
Notebook: ['00_setup_e_pulizia.ipynb', '09_genera_tracce_SAM2_7clip.ipynb', '10_E1_detection_confronto.ipynb', '11_E2_tracking_confronto.ipynb', '12_E3_correzione_coordinate.ipynb', '13_demo_squadre_mappa2D.ipynb']
